# Phase 3 Step 1b -- full-corpus pseudo-labeling, shard 0/2

Scores every report in the 4,407-study corpus against the host rubric with the bake-off winner
**Qwen3-4B-Instruct-2507** (gold macro AUC 0.8613, 100% answer rate, run 2 -- see NOTES.md
2026-08-25). One prompt per (study, label): this shard owns 26448 prompts.

Sharding follows the Phase 2 prep pattern: identical kernels differing only in `SHARD_INDEX`,
each owning an exact contiguous index range over the *sorted* corpus UID list, so any lost or
timed-out shard re-runs alone and reproduces exactly its own studies. Progress is rewritten to
`scores_shard0.csv` every 25 batches (~every few minutes), so even a killed session keeps its
completed work. `_SHARD_0_COMPLETE` is written only on success -- the consumer kernel treats
a shard without it as failed regardless of what files survived.

Rules compliance: report text never leaves this notebook and is never sent to a hosted API.
The harness functions are copied verbatim from the bake-off kernel v10, where they were
validated on real Kaggle hardware (run 2: zero OOM retries, batched scores bit-equal to solo).


In [ ]:
import glob, json, os, shutil, sys, time

GIT_SHA = '0b80382'
SHARD_INDEX = 0
N_SHARDS = 2
MODEL_REPO = 'Qwen/Qwen3-4B-Instruct-2507'
TOKEN_BUDGET = 16384
MAX_BATCH = 64
SAVE_EVERY_N_BATCHES = 25
OUT_DIR = '/kaggle/working'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')
print('src/knee mounted from', SRC)


In [ ]:
import torch
assert torch.cuda.is_available(), 'no GPU attached -- pick GPU T4 x2 before Save & Run All'
n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    major, minor = torch.cuda.get_device_capability(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, compute capability sm_{major}{minor}')
    # P100 (sm_60) sessions have been assigned despite selecting T4x2. Fail fast
    # and loud here rather than deep inside a CUDA kernel launch.
    assert (major, minor) >= (7, 0), (
        f'GPU {i} is sm_{major}{minor}, too old for fp16 inference on this torch build. '
        'Kaggle likely assigned a P100 -- stop this run, pick GPU T4 x2 explicitly in the '
        'notebook editor accelerator settings, then Save Version -> Save & Run All (Commit).')
print(f'{n_gpus} GPU(s) OK')

import transformers
print('transformers', transformers.__version__, '| torch', torch.__version__)


In [ ]:
import numpy as np
import pandas as pd

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
n_all = len(all_uids)
assert n_all == 4407, f'expected 4407 studies, got {n_all}'

# Exact contiguous index ranges over the sorted UID list -- the same partition
# contract as the Phase 2 prep shards (near-even halves covering every study
# exactly once; re-running shard i always reproduces shard i's studies).
bounds = [i * n_all // N_SHARDS for i in range(N_SHARDS + 1)]
LO, HI = bounds[SHARD_INDEX], bounds[SHARD_INDEX + 1]
shard_uids = all_uids[LO:HI]
assert len(set(shard_uids)) == len(shard_uids)

reports_map = dict(zip(train_df['StudyInstanceUID'].astype(str), train_df['Report']))
missing = [u for u in shard_uids if not isinstance(reports_map.get(u), str)]
assert not missing, f'{len(missing)} studies in this shard have no Report text'

print(f'shard {SHARD_INDEX}: studies [{LO}, {HI}) of [0, {n_all}) -> '
      f'{len(shard_uids)} studies')


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_REPO, padding_side='left')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO, torch_dtype=torch.float16, device_map='auto')
model.eval()
print('model loaded:', MODEL_REPO)


In [ ]:
from types import SimpleNamespace

from knee.infer import LABEL_COLUMNS
from knee.reports import build_label_prompt, score_from_top_logprobs

# --- harness copied verbatim from knee-phase3-bakeoff v10 (validated on T4x2,
# run 2: 696/696 answered, 0 OOM retries) -- do not diverge lightly ---
_YES_FORMS = ('Yes', 'yes', ' Yes', ' yes', 'YES')
_NO_FORMS = ('No', 'no', ' No', ' no', 'NO')


def single_token_forms(tk, forms):
    """{token_id: surface_form} for the forms this tokenizer encodes as exactly
    one token. Multi-token forms are unusable: the score is read at a single
    answer position, so a form spanning two tokens has no logit there."""
    out = {}
    for form in forms:
        ids = tk.encode(form, add_special_tokens=False)
        if len(ids) == 1:
            out[ids[0]] = form
    return out


def render(tk, prompt):
    """Apply the chat template so the model's first generated token is the
    answer. enable_thinking=False is REQUIRED for Qwen3-8B: its default
    template leaves the assistant turn open, so the first token would open a
    <think> block and every answer would score NaN. Gemma's template rejects
    the kwarg, hence try/except rather than an assumption either way."""
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        return tk.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                      enable_thinking=False)
    except TypeError:
        return tk.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def score_batch(md, tk, rendered, yes_ids, no_ids):
    """One forward pass over a batch of already-rendered prompts; returns the
    per-prompt {token_id: Logprob-like} mapping score_from_top_logprobs takes.

    logits_to_keep=1 is load-bearing, not an optimisation. Without it the model
    materialises logits at every position -- at batch 8 x seq 1849 x 151936
    vocab that is 4.49 GB in fp16, which is precisely what OOMed the 8B on the
    first run. Only the last position is ever read. Verified locally that the
    answer-position logits are bit-identical either way.
    """
    enc = tk(rendered, return_tensors='pt', padding=True, add_special_tokens=False,
             truncation=True, max_length=4096).to(md.device)
    try:
        out = md(**enc, logits_to_keep=1)
    except TypeError:
        try:
            out = md(**enc, num_logits_to_keep=1)
        except TypeError:
            out = md(**enc)
    logits = out.logits[:, -1, :].float()
    logprobs = torch.log_softmax(logits, dim=-1)
    return [{tid: SimpleNamespace(decoded_token=form, logprob=logprobs[row, tid].item())
              for tid, form in {**yes_ids, **no_ids}.items()}
            for row in range(logprobs.shape[0])]


def make_batches(order, lengths, token_budget, max_batch):
    """Greedy batches over length-sorted keys, capped so that
    (batch size x longest member) stays within token_budget. Sorting first
    keeps each batch near-uniform, so the padding cost stays small."""
    batches, cur = [], []
    for key in order:
        trial = cur + [key]
        width = max(lengths[k] for k in trial) * len(trial)
        if cur and (width > token_budget or len(trial) > max_batch):
            batches.append(cur)
            cur = [key]
        else:
            cur = trial
    if cur:
        batches.append(cur)
    return batches


yes_ids = single_token_forms(tok, _YES_FORMS)
no_ids = single_token_forms(tok, _NO_FORMS)
assert yes_ids and no_ids, 'tokenizer offers no single-token Yes/No form'
print('yes tokens:', list(yes_ids.values()), '| no tokens:', list(no_ids.values()))


In [ ]:
SCORE_COLS = [f'score_{l}' for l in LABEL_COLUMNS]
WEIGHT_COLS = [f'weight_{l}' for l in LABEL_COLUMNS]


def save_scores(scores, weights):
    df = pd.DataFrame(scores, columns=SCORE_COLS)
    wdf = pd.DataFrame(weights, columns=WEIGHT_COLS)
    out = pd.concat([pd.DataFrame({'StudyInstanceUID': shard_uids}), df, wdf], axis=1)
    out.to_csv(f'{OUT_DIR}/scores_shard0.csv', index=False)


PROMPTS = [(si, li) for si in range(len(shard_uids)) for li in range(len(LABEL_COLUMNS))]
t_build = time.time()
rendered = {(si, li): render(tok, build_label_prompt(
    reports_map[shard_uids[si]], LABEL_COLUMNS[li])) for si, li in PROMPTS}
lengths = {k: len(tok(v, add_special_tokens=False)['input_ids']) for k, v in rendered.items()}
order = sorted(PROMPTS, key=lambda k: lengths[k])
lens = list(lengths.values())
batches = make_batches(order, lengths, TOKEN_BUDGET, MAX_BATCH)
print(f'{len(PROMPTS)} prompts | tokens min {min(lens)} median '
      f'{sorted(lens)[len(lens)//2]} max {max(lens)} | {len(batches)} batches, '
      f'sizes {min(len(b) for b in batches)}-{max(len(b) for b in batches)} '
      f'(built in {time.time()-t_build:.0f}s)', flush=True)

scores = np.full((len(shard_uids), len(LABEL_COLUMNS)), np.nan)
weights = np.zeros_like(scores)
n_oom_retries = 0
t0 = time.time()
for bi, batch in enumerate(batches):
    queue, done = [batch], []
    while queue:
        chunk = queue.pop(0)
        try:
            done.append((chunk, score_batch(model, tok,
                                            [rendered[k] for k in chunk], yes_ids, no_ids)))
        except torch.cuda.OutOfMemoryError:
            if len(chunk) == 1:
                raise
            n_oom_retries += 1
            torch.cuda.empty_cache()
            mid = len(chunk) // 2
            queue[:0] = [chunk[:mid], chunk[mid:]]
    for chunk, mappings in done:
        for key, mapping in zip(chunk, mappings):
            score, weight = score_from_top_logprobs(mapping)
            si, li = key
            scores[si, li] = np.nan if score is None else score
            weights[si, li] = weight
    if (bi + 1) % SAVE_EVERY_N_BATCHES == 0 or bi == len(batches) - 1:
        save_scores(scores, weights)
        done_frac = (bi + 1) / len(batches)
        eta_min = (time.time() - t0) / done_frac * (1 - done_frac) / 60
        answered = int((~np.isnan(scores)).sum())
        print(f'  batch {bi+1}/{len(batches)} ({time.time()-t0:.0f}s, '
              f'eta ~{eta_min:.0f}min) answered {answered}/{len(PROMPTS)} '
              f'-> saved', flush=True)

elapsed = time.time() - t0
answered = int((~np.isnan(scores)).sum())
assert answered == len(PROMPTS), (
    f'{answered}/{len(PROMPTS)} scored -- incomplete shard must NOT claim completion')

manifest = {
    'shard_index': SHARD_INDEX, 'n_shards': N_SHARDS, 'uid_range': [int(LO), int(HI)],
    'git_sha': GIT_SHA, 'model_repo': MODEL_REPO,
    'n_studies': int(len(shard_uids)), 'n_prompts': int(len(PROMPTS)),
    'n_batches': int(len(batches)), 'elapsed_s': round(elapsed, 1),
    'n_oom_retries': int(n_oom_retries),
    'answer_rate': answered / len(PROMPTS),
    'mean_score': float(np.nanmean(scores)), 'max_score': float(np.nanmax(scores)),
    'median_prompt_tokens': int(sorted(lens)[len(lens) // 2]),
    'max_prompt_tokens': int(max(lens)),
}
with open(f'{OUT_DIR}/manifest_shard{SHARD_INDEX}.json', 'w') as f:
    json.dump(manifest, f, indent=2)
# written LAST on purpose: its existence is the consumer's success signal
with open(f'{OUT_DIR}/_SHARD_{SHARD_INDEX}_COMPLETE', 'w') as f:
    f.write('ok\n')

print(json.dumps(manifest, indent=2))
print(f'SHARD {SHARD_INDEX} COMPLETE: {len(shard_uids)} studies, '
      f'{elapsed:.0f}s, {n_oom_retries} OOM retries', flush=True)
